In [2]:
import os
import json
import tqdm
import numpy as np
import pandas as pd
from IPython.display import display

In [3]:
dataset_mapping = {
    "20250804_myekyc_fullday-annotated_filtered_annotated_index_annotation_mykadfront_genuine": "MyDigitalID_Genuine",
    "20250804_myekyc_fullday-annotated_filtered_annotated_index_annotation_mykadfront_print": "MyDigitalID_Print",
    # "20250804_myekyc_fullday-annotated_filtered_annotated_index_annotation_mykadfront_tamper": "MyDigitalID_Tamper",
    "20250805_myekyc_fullday-annotated_index_annotation_mykadfront_cutout": "FullEKYC_20250805_CutOut",
    "20250805_myekyc_fullday-annotated_index_annotation_mykadfront_genuine": "FullEKYC_20250805_Genuine",
    "20250805_myekyc_fullday-annotated_index_annotation_mykadfront_printed": "FullEKYC_20250805_Printed",
    "20250805_myekyc_fullday-annotated_index_annotation_mykadfront_replay": "FullEKYC_20250805_Replay",
    "20250806_myekyc_owntester-annotated_index_annotation_mykadfront_v3_replay": "FullEKYC_20250806_Replay",
    "20250806_myekyc_owntester-annotated_index_annotation_mykadfront_v3_print": "FullEKYC_20250806_Print",
    "20250806_myekyc_owntester-annotated_index_annotation_mykadfront_v3_genuine": "FullEKYC_20250806_Genuine",
    # "20250806_myekyc_owntester-annotated_index_annotation_mykadfront_v3_tamper": "FullEKYC_20250806_Tamper",
    "20250807_genuine-annotated_index_annotation": "FullEKYC_20250807_Genuine",
    "20250808_genuine&cutout-annotated_index_annotation_genuine": "FullEKYC_20250808_Genuine_1",
    "20250808_genuine&cutout-annotated_index_annotation_print_cutout": "FullEKYC_20250808_CutOut",
    "20250808_replay_tablet-annotated_index_annotation_genuine": "FullEKYC_20250808_Genuine_2",
    "20250808_replay_tablet-annotated_index_annotation_replay": "FullEKYC_20250808_Replay",
    "20250808_tamper_face-annotated_index_annotation_genuine": "FullEKYC_20250808_Genuine_3",
    "20250811_genuine_ekyc-filtered_index_annotation": "FullEKYC_20250811_Genuine",
    "20250811_Replay_FullPage-filtered_index_annotation_v2_print": "FullEKYC_20250811_Print",
    "20250811_Replay_FullPage-filtered_index_annotation_v2_replay": "FullEKYC_20250811_Replay",
    "20250813_grayscale-annotated_annotated_index_annotation_mykadfront_replay": "FullEKYC_20250813_Replay",
    "20250813_grayscale-annotated_annotated_index_annotation_mykadfront_print": "FullEKYC_20250813_Print",
    "20250813_grayscale-annotated_annotated_index_annotation_mykadfront_grayscale": "FullEKYC_20250813_Grayscale",
    "20250813_wiseai_myid_test-dry_run-index_annotation_mykadfront_v4_genuine": "MyIDTestDryRun_20250813_Genuine",
    "20250813_wiseai_myid_test-dry_run-index_annotation_mykadfront_v4_print": "MyIDTestDryRun_20250813_Print",
    "20250813_wiseai_myid_test-dry_run-index_annotation_mykadfront_v4_replay": "MyIDTestDryRun_20250813_Replay",
    "genuine_with_background_test_plan_subject-genuine_with_background_test_plan_subject": "1.1_Genuine_w_Background",
    "grayscale_print_test_plan_subject-grayscale_print_test_plan_subject": "1.2 Grayscale Print Test Plan",
    "color_print_test_plan_subject-color_print_test_plan_subject": "1.3 Color Print Test Plan",
    "grayscale_print_cutout_test_plan_subject-grayscale_print_cutout_test_plan_subject": "1.4_GreyCutOut",
    "colorprint_cutout_test_plan_subject-colorprint_cutout_test_plan_subject": "1.5_ColorCutOut",
    "color_print_with_background2_test_plan_subject-color_print_with_background2_test_plan_subject": "1.6.1 Color Print with Background 2",
    "replay_monitor_test_plan_subject-replay_monitor_test_plan_subject": "1.10 Replay Monitor Test Plan",
    "replay_mobile_test_plan_subject-replay_mobile_test_plan_subject": "1.11 Replay Mobile Test Plan",
    "replay_tablet_test_plan_subject-replay_tablet_test_plan_subject": "1.12 Replay Tablet Test Plan",
    "batch_issue_20250910_bmmb_both_general-index_annotation_mykadfront_v3": "BMMB_Issue",
    "batch_production_20250830_20250903_myid-annotated_index_annotation_mykadfront_v2_genuine": "MyDigitalID_20250830_20250903_Genuine",
    "batch_production_20250830_20250903_myid-annotated_index_annotation_mykadfront_v2_print": "MyDigitalID_20250830_20250903_Print",
    "batch_production_20250830_20250903_myid-annotated_index_annotation_mykadfront_v2_replay": "MyDigitalID_20250830_20250903_Replay",
    "batch_production_20240513_20240519_snt_random-batch_production_20240513_20240519_snt_random": "SNT_20240513_20240519",
    "batch_production_202208_redone-index_annotation": "RedOne_202208"
}

dataset_mapping_corner = {
    "20250804_myekyc_fullday-annotated_filtered_annotated_index_annotation_mykadfront_genuine_corners": "MyDigitalID_Genuine",
    "20250804_myekyc_fullday-annotated_filtered_annotated_index_annotation_mykadfront_prin_cornerst": "MyDigitalID_Print",
    # "20250804_myekyc_fullday-annotated_filtered_annotated_index_annotation_mykadfront_tamper_corners": "MyDigitalID_Tamper",
    "20250805_myekyc_fullday-annotated_index_annotation_mykadfront_cutout_corners": "FullEKYC_20250805_CutOut",
    "20250805_myekyc_fullday-annotated_index_annotation_mykadfront_genuine_corners": "FullEKYC_20250805_Genuine",
    "20250805_myekyc_fullday-annotated_index_annotation_mykadfront_printed_corners": "FullEKYC_20250805_Printed",
    "20250805_myekyc_fullday-annotated_index_annotation_mykadfront_replay_corners": "FullEKYC_20250805_Replay",
    "20250806_myekyc_owntester-annotated_index_annotation_mykadfront_v3_replay_corners": "FullEKYC_20250806_Replay",
    "20250806_myekyc_owntester-annotated_index_annotation_mykadfront_v3_print_corners": "FullEKYC_20250806_Print",
    "20250806_myekyc_owntester-annotated_index_annotation_mykadfront_v3_genuine_corners": "FullEKYC_20250806_Genuine",
    # "20250806_myekyc_owntester-annotated_index_annotation_mykadfront_v3_tamper_corners": "FullEKYC_20250806_Tamper",
    "20250807_genuine-annotated_index_annotation_corners": "FullEKYC_20250807_Genuine",
    "20250808_genuine&cutout-annotated_index_annotation_genuine_corners": "FullEKYC_20250808_Genuine_1",
    "20250808_genuine&cutout-annotated_index_annotation_print_cutout_corners": "FullEKYC_20250808_CutOut",
    "20250808_replay_tablet-annotated_index_annotation_genuine_corners": "FullEKYC_20250808_Genuine_2",
    "20250808_replay_tablet-annotated_index_annotation_replay_corners": "FullEKYC_20250808_Replay",
    "20250808_tamper_face-annotated_index_annotation_genuine_corners": "FullEKYC_20250808_Genuine_3",
    "20250811_genuine_ekyc-filtered_index_annotation_corners": "FullEKYC_20250811_Genuine",
    "20250811_Replay_FullPage-filtered_index_annotation_v2_print_corners": "FullEKYC_20250811_Print",
    "20250811_Replay_FullPage-filtered_index_annotation_v2_replay_corners": "FullEKYC_20250811_Replay",
    "20250813_grayscale-annotated_annotated_index_annotation_mykadfront_replay_corners": "FullEKYC_20250813_Replay",
    "20250813_grayscale-annotated_annotated_index_annotation_mykadfront_print_corners": "FullEKYC_20250813_Print",
    "20250813_grayscale-annotated_annotated_index_annotation_mykadfront_grayscale_corners": "FullEKYC_20250813_Grayscale",
    "20250813_wiseai_myid_test-dry_run-index_annotation_mykadfront_v4_genuine_corners": "MyIDTestDryRun_20250813_Genuine",
    "20250813_wiseai_myid_test-dry_run-index_annotation_mykadfront_v4_print_corners": "MyIDTestDryRun_20250813_Print",
    "20250813_wiseai_myid_test-dry_run-index_annotation_mykadfront_v4_replay_corners": "MyIDTestDryRun_20250813_Replay",
    "genuine_with_background_test_plan_subject_corners-genuine_with_background_test_plan_subject_corners": "1.1_Genuine_w_Background",
    "grayscale_print_test_plan_subject_corners-grayscale_print_test_plan_subject_corners": "1.2 Grayscale Print Test Plan",
    "color_print_test_plan_subject_corners-color_print_test_plan_subject_corners": "1.3 Color Print Test Plan",
    "grayscale_print_cutout_test_plan_subject_corners-grayscale_print_cutout_test_plan_subject_corners": "1.4_GreyCutOut",
    "colorprint_cutout_test_plan_subject_corners-colorprint_cutout_test_plan_subject_corners": "1.5_ColorCutOut",
    "color_print_with_background2_test_plan_subject_corners-color_print_with_background2_test_plan_subject_corners": "1.6.1 Color Print with Background 2",
    "replay_monitor_test_plan_subject_corners-replay_monitor_test_plan_subject_corners": "1.10 Replay Monitor Test Plan",
    "replay_mobile_test_plan_subject_corners-replay_mobile_test_plan_subject_corners": "1.11 Replay Mobile Test Plan",
    "replay_tablet_test_plan_subject_corners-replay_tablet_test_plan_subject_corners": "1.12 Replay Tablet Test Plan",
    "batch_issue_20250910_bmmb_both_general-index_annotation_mykadfront_v3_corners": "BMMB_Issue",
    "batch_production_20250830_20250903_myid-annotated_index_annotation_mykadfront_v2_genuine_corners": "MyDigitalID_20250830_20250903_Genuine",
    "batch_production_20250830_20250903_myid-annotated_index_annotation_mykadfront_v2_print_corners": "MyDigitalID_20250830_20250903_Print",
    "batch_production_20250830_20250903_myid-annotated_index_annotation_mykadfront_v2_replay_corners": "MyDigitalID_20250830_20250903_Replay"
}

dataset_mapping_topedge = {
    "20250804_myekyc_fullday-annotated_filtered_annotated_index_annotation_mykadfront_genuine_topedge": "MyDigitalID_Genuine",
    "20250804_myekyc_fullday-annotated_filtered_annotated_index_annotation_mykadfront_print_topedge": "MyDigitalID_Print",
    # "20250804_myekyc_fullday-annotated_filtered_annotated_index_annotation_mykadfront_tamper_topedge": "MyDigitalID_Tamper",
    "20250805_myekyc_fullday-annotated_index_annotation_mykadfront_cutout_topedge": "FullEKYC_20250805_CutOut",
    "20250805_myekyc_fullday-annotated_index_annotation_mykadfront_genuine_topedge": "FullEKYC_20250805_Genuine",
    "20250805_myekyc_fullday-annotated_index_annotation_mykadfront_printed_topedge": "FullEKYC_20250805_Printed",
    "20250805_myekyc_fullday-annotated_index_annotation_mykadfront_replay_topedge": "FullEKYC_20250805_Replay",
    "20250806_myekyc_owntester-annotated_index_annotation_mykadfront_v3_replay_topedge": "FullEKYC_20250806_Replay",
    "20250806_myekyc_owntester-annotated_index_annotation_mykadfront_v3_print_topedge": "FullEKYC_20250806_Print",
    "20250806_myekyc_owntester-annotated_index_annotation_mykadfront_v3_genuine_topedge": "FullEKYC_20250806_Genuine",
    # "20250806_myekyc_owntester-annotated_index_annotation_mykadfront_v3_tamper_topedge": "FullEKYC_20250806_Tamper",
    "20250807_genuine-annotated_index_annotation_topedge": "FullEKYC_20250807_Genuine",
    "20250808_genuine&cutout-annotated_index_annotation_genuine_topedge": "FullEKYC_20250808_Genuine_1",
    "20250808_genuine&cutout-annotated_index_annotation_print_cutout_topedge": "FullEKYC_20250808_CutOut",
    "20250808_replay_tablet-annotated_index_annotation_genuine_topedge": "FullEKYC_20250808_Genuine_2",
    "20250808_replay_tablet-annotated_index_annotation_replay_topedge": "FullEKYC_20250808_Replay",
    "20250808_tamper_face-annotated_index_annotation_genuine_topedge": "FullEKYC_20250808_Genuine_3",
    "20250811_genuine_ekyc-filtered_index_annotation_topedge": "FullEKYC_20250811_Genuine",
    "20250811_Replay_FullPage-filtered_index_annotation_v2_print_topedge": "FullEKYC_20250811_Print",
    "20250811_Replay_FullPage-filtered_index_annotation_v2_replay_topedge": "FullEKYC_20250811_Replay",
    "20250813_grayscale-annotated_annotated_index_annotation_mykadfront_replay_topedge": "FullEKYC_20250813_Replay",
    "20250813_grayscale-annotated_annotated_index_annotation_mykadfront_print_topedge": "FullEKYC_20250813_Print",
    "20250813_grayscale-annotated_annotated_index_annotation_mykadfront_grayscale_topedge": "FullEKYC_20250813_Grayscale",
    "20250813_wiseai_myid_test-dry_run-index_annotation_mykadfront_v4_genuine_topedge": "MyIDTestDryRun_20250813_Genuine",
    "20250813_wiseai_myid_test-dry_run-index_annotation_mykadfront_v4_print_topedge": "MyIDTestDryRun_20250813_Print",
    "20250813_wiseai_myid_test-dry_run-index_annotation_mykadfront_v4_replay_topedge": "MyIDTestDryRun_20250813_Replay",
    "genuine_with_background_test_plan_subject_landmarks-genuine_with_background_test_plan_subject_topedge": "1.1_Genuine_w_Background",
    "grayscale_print_test_plan_subject_landmarks-grayscale_print_test_plan_subject_topedge": "1.2 Grayscale Print Test Plan",
    "color_print_test_plan_subject_landmarks-color_print_test_plan_subject_topedge": "1.3 Color Print Test Plan",
    "grayscale_print_cutout_test_plan_subject_landmarks-grayscale_print_cutout_test_plan_subject_topedge": "1.4_GreyCutOut",
    "colorprint_cutout_test_plan_subject_landmarks-colorprint_cutout_test_plan_subject_topedge": "1.5_ColorCutOut",
    "color_print_with_background2_test_plan_subject_landmarks-color_print_with_background2_test_plan_subject_topedge": "1.6.1 Color Print with Background 2",
    "replay_monitor_test_plan_subject_landmarks-replay_monitor_test_plan_subject_topedge": "1.10 Replay Monitor Test Plan",
    "replay_mobile_test_plan_subject_landmarks-replay_mobile_test_plan_subject_topedge": "1.11 Replay Mobile Test Plan",
    "replay_tablet_test_plan_subject_landmarks-replay_tablet_test_plan_subject_topedge": "1.12 Replay Tablet Test Plan",
    "batch_issue_20250910_bmmb_both_general-index_annotation_mykadfront_v3_topedge": "BMMB_Issue",
    "batch_production_20250830_20250903_myid-annotated_index_annotation_mykadfront_v2_genuine_topedge": "MyDigitalID_20250830_20250903_Genuine",
    "batch_production_20250830_20250903_myid-annotated_index_annotation_mykadfront_v2_print_topedge": "MyDigitalID_20250830_20250903_Print",
    "batch_production_20250830_20250903_myid-annotated_index_annotation_mykadfront_v2_replay_topedge": "MyDigitalID_20250830_20250903_Replay",
    "batch_production_20240513_20240519_snt_random-batch_production_20240513_20240519_snt_random_topedge": "SNT_20240513_20240519",
    "batch_production_202208_redone-index_annotation_topedge": "RedOne_202208"
}

dataset_mapping_kpmlogo = {
    "20250804_myekyc_fullday-annotated_filtered_annotated_index_annotation_mykadfront_genuine_kpmlogo": "MyDigitalID_Genuine",
    "20250804_myekyc_fullday-annotated_filtered_annotated_index_annotation_mykadfront_print_kpmlogo": "MyDigitalID_Print",
    # "20250804_myekyc_fullday-annotated_filtered_annotated_index_annotation_mykadfront_tamper_kpmlogo": "MyDigitalID_Tamper",
    "20250805_myekyc_fullday-annotated_index_annotation_mykadfront_cutout_kpmlogo": "FullEKYC_20250805_CutOut",
    "20250805_myekyc_fullday-annotated_index_annotation_mykadfront_genuine_kpmlogo": "FullEKYC_20250805_Genuine",
    "20250805_myekyc_fullday-annotated_index_annotation_mykadfront_printed_kpmlogo": "FullEKYC_20250805_Printed",
    "20250805_myekyc_fullday-annotated_index_annotation_mykadfront_replay_kpmlogo": "FullEKYC_20250805_Replay",
    "20250806_myekyc_owntester-annotated_index_annotation_mykadfront_v3_replay_kpmlogo": "FullEKYC_20250806_Replay",
    "20250806_myekyc_owntester-annotated_index_annotation_mykadfront_v3_print_kpmlogo": "FullEKYC_20250806_Print",
    "20250806_myekyc_owntester-annotated_index_annotation_mykadfront_v3_genuine_kpmlogo": "FullEKYC_20250806_Genuine",
    # "20250806_myekyc_owntester-annotated_index_annotation_mykadfront_v3_tamper_kpmlogo": "FullEKYC_20250806_Tamper",
    "20250807_genuine-annotated_index_annotation_kpmlogo": "FullEKYC_20250807_Genuine",
    "20250808_genuine&cutout-annotated_index_annotation_genuine_kpmlogo": "FullEKYC_20250808_Genuine_1",
    "20250808_genuine&cutout-annotated_index_annotation_print_cutout_kpmlogo": "FullEKYC_20250808_CutOut",
    "20250808_replay_tablet-annotated_index_annotation_genuine_kpmlogo": "FullEKYC_20250808_Genuine_2",
    "20250808_replay_tablet-annotated_index_annotation_replay_kpmlogo": "FullEKYC_20250808_Replay",
    "20250808_tamper_face-annotated_index_annotation_genuine_kpmlogo": "FullEKYC_20250808_Genuine_3",
    "20250811_genuine_ekyc-filtered_index_annotation_kpmlogo": "FullEKYC_20250811_Genuine",
    "20250811_Replay_FullPage-filtered_index_annotation_v2_print_kpmlogo": "FullEKYC_20250811_Print",
    "20250811_Replay_FullPage-filtered_index_annotation_v2_replay_kpmlogo": "FullEKYC_20250811_Replay",
    "20250813_grayscale-annotated_annotated_index_annotation_mykadfront_replay_kpmlogo": "FullEKYC_20250813_Replay",
    "20250813_grayscale-annotated_annotated_index_annotation_mykadfront_print_kpmlogo": "FullEKYC_20250813_Print",
    "20250813_grayscale-annotated_annotated_index_annotation_mykadfront_grayscale_kpmlogo": "FullEKYC_20250813_Grayscale",
    "20250813_wiseai_myid_test-dry_run-index_annotation_mykadfront_v4_genuine_kpmlogo": "MyIDTestDryRun_20250813_Genuine",
    "20250813_wiseai_myid_test-dry_run-index_annotation_mykadfront_v4_print_kpmlogo": "MyIDTestDryRun_20250813_Print",
    "20250813_wiseai_myid_test-dry_run-index_annotation_mykadfront_v4_replay_kpmlogo": "MyIDTestDryRun_20250813_Replay",
    "genuine_with_background_test_plan_subject_landmarks-genuine_with_background_test_plan_subject_kpmlogo": "1.1_Genuine_w_Background",
    "grayscale_print_test_plan_subject_landmarks-grayscale_print_test_plan_subject_kpmlogo": "1.2 Grayscale Print Test Plan",
    "color_print_test_plan_subject_landmarks-color_print_test_plan_subject_kpmlogo": "1.3 Color Print Test Plan",
    "grayscale_print_cutout_test_plan_subject_landmarks-grayscale_print_cutout_test_plan_subject_kpmlogo": "1.4_GreyCutOut",
    "colorprint_cutout_test_plan_subject_landmarks-colorprint_cutout_test_plan_subject_kpmlogo": "1.5_ColorCutOut",
    "color_print_with_background2_test_plan_subject_landmarks-color_print_with_background2_test_plan_subject_kpmlogo": "1.6.1 Color Print with Background 2",
    "replay_monitor_test_plan_subject_landmarks-replay_monitor_test_plan_subject_kpmlogo": "1.10 Replay Monitor Test Plan",
    "replay_mobile_test_plan_subject_landmarks-replay_mobile_test_plan_subject_kpmlogo": "1.11 Replay Mobile Test Plan",
    "replay_tablet_test_plan_subject_landmarks-replay_tablet_test_plan_subject_kpmlogo": "1.12 Replay Tablet Test Plan",
    "batch_issue_20250910_bmmb_both_general-index_annotation_mykadfront_v3_kpmlogo": "BMMB_Issue",
    "batch_production_20250830_20250903_myid-annotated_index_annotation_mykadfront_v2_genuine_kpmlogo": "MyDigitalID_20250830_20250903_Genuine",
    "batch_production_20250830_20250903_myid-annotated_index_annotation_mykadfront_v2_print_kpmlogo": "MyDigitalID_20250830_20250903_Print",
    "batch_production_20250830_20250903_myid-annotated_index_annotation_mykadfront_v2_replay_kpmlogo": "MyDigitalID_20250830_20250903_Replay"
}

mapping_dicts = {
    "baseline": dataset_mapping,
    "corner": dataset_mapping_corner,
    "top_edge": dataset_mapping_topedge,
    "kpmlogo": dataset_mapping_kpmlogo,
}

# Function to load dataset results for a model
def load_results(model_path):
    datasets = {}
    for dataset_folder in os.listdir(model_path):
        dataset_path = os.path.join(model_path, dataset_folder, f"{dataset_folder}.csv")
        if os.path.exists(dataset_path):
            datasets[dataset_folder] = pd.read_csv(dataset_path)
    return datasets

def normalize_datasets(datasets, mapping):
    """
    Rename dataset keys using provided mapping
    """
    normalized = {}
    for raw_name, df in datasets.items():
        if raw_name in mapping:
            normalized_name = mapping[raw_name]
            normalized[normalized_name] = df
        else:
            print(f"Warning: {raw_name} not in mapping")
    return normalized

In [ ]:
new_baseline_model_path = "/mnt3/auto-ekyc/idrecapture/artifacts/parallel/Ench21_v1_20251010-1106/infer_results"
new_compare1_model_path = "/mnt3/auto-ekyc/idrecapture/artifacts/parallel/PHL_Ench1_v2_20251109-1400/infer_results"
new_compare2_model_path = "/mnt3/auto-ekyc/idrecapture/artifacts/parallel/PHL_Ench1_v3_20251113-0106/infer_results"

# --- Function to calculate APCER/BPCER ---
def evaluate_model(model_path, model_name):
    results_raw = load_results(model_path)
    results = normalize_datasets(results_raw, mapping_dicts["baseline"])
    summary = []

    for dataset_name, df in results.items():
        # Assume columns: id, ground_truth, pred
        combined = df.copy()

        # Errors
        combined["error"] = (combined["ypred"] != combined["label"]).astype(int)

        # Bona fide = 0, Attack = 1 (adjust if your labels differ!)
        bona_fide = combined[combined["label"] == 0]
        attack = combined[combined["label"] == 1]

        apcer = (attack["ypred"] == 0).mean() if len(attack) > 0 else None
        bpcer = (bona_fide["ypred"] == 1).mean() if len(bona_fide) > 0 else None

        summary.append({
            "dataset": dataset_name,
            "model": model_name,
            "APCER": apcer,
            "BPCER": bpcer
        })

    return pd.DataFrame(summary)

summary_model_baseline1 = evaluate_model(new_baseline_model_path, "Baseline")
summary_model_compare1 = evaluate_model(new_compare1_model_path, "IdRecapture Model PHL Train")
summary_model_compare2 = evaluate_model(new_compare2_model_path, "IdRecapture Model PHL Train + PHILSIGN")

# Load results from both models
new_baseline_datasets_raw = load_results(new_baseline_model_path)
# new_compare1_datasets_raw = load_results(summary_model_compare1)

new_baseline_datasets = normalize_datasets(new_baseline_datasets_raw, mapping_dicts["baseline"])
# new_compare1_datasets = normalize_datasets(new_compare1_datasets_raw, mapping_dicts["baseline"])

def merge_evaluate(baseline_datasets, compare_dataset, merge_type, model_name, threshold_list):
    summary = []
    display_name = model_name
    for threshold in threshold_list:
        for dataset_name in compare_dataset:
            if dataset_name not in baseline_datasets:
                # print(f"Skipped: {dataset_name}")
                continue # skip datasets not present in both models

            baseline_results = baseline_datasets[dataset_name]
            compare_results = compare_dataset[dataset_name]

            # Align by row index (assumes same ordering and rows)
            combined =  compare_results.copy()

            compare_results["ypred"] = ( compare_results["ypred_raw"] > threshold).astype(int)
            baseline_results["ypred"] = (baseline_results["ypred_raw"] > 0.5).astype(int)

            if merge_type == "max":
                combined["ypred_final"] =  compare_results["ypred"].combine(baseline_results["ypred"], max)
                model_name = display_name + f"_max_{threshold}"
            elif merge_type == "min":
                combined["ypred_final"] =  compare_results["ypred"].combine(baseline_results["ypred"], min)
                model_name = display_name + f"_min_{threshold}"

            bona_fide = combined[combined["label"] == 0]
            attack = combined[combined["label"] == 1]

            apcer = (attack["ypred_final"] == 0).mean() if len(attack) > 0 else None
            bpcer = (bona_fide["ypred_final"] == 1).mean() if len(bona_fide) > 0 else None
            summary.append({
                "dataset": dataset_name,
                "model": model_name,
                "APCER": apcer,
                "BPCER": bpcer
            })
    return summary


# merged_summary_df = pd.concat([summary_model_baseline1, summary_df_cornerbase_max, summary_df_topedgebase_max, summary_df_kpmlogobase_max, summary_df_kpmlogobase_eff_224_max, summary_df_kpmlogobase_eff_max], ignore_index=True) # without min
# merged_summary_df = pd.concat([summary_model_baseline1, new_summary_model_baseline1, new_summary_df_topedgebase_max, new_summary_df_kpmlogobase_eff_max], ignore_index=True) # without min
merged_summary_df = pd.concat([summary_model_baseline1, summary_model_compare1, summary_model_compare2], ignore_index=True) # without min

# display(merged_summary_df)

# --- Filter merged results ---
# Pivot back into wide format for plotting
wide_filtered = merged_summary_df.pivot(index="dataset", columns="model", values=["APCER","BPCER"])
wide_filtered = wide_filtered.swaplevel(axis=1).sort_index(axis=1, level=0)

styled = (
    wide_filtered.style
    .background_gradient(
        cmap="Reds",   # white = low, red = high
        axis=None
    )
    .format("{:.3f}")
    .applymap(lambda v: "background-color: gray" if pd.isna(v) else "")
)

pd.set_option("display.max_columns", None)  # show all columns
pd.set_option("display.width", 200)

display(styled)

/tmp/ipykernel_1525527/962026456.py:96: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  wide_filtered.style
